In [35]:
import sys
import os
from pathlib import Path

package_path = Path(os.path.abspath("")).parent


In [36]:
from section_identification.preprocess import preprocess_image
from section_identification.filtering import filtering
from section_identification.section_detector import automatic_identification

apply_filtering = True

image1 = package_path / "images/example1.png"
image2 = package_path / "images/example2.png"
image3 = package_path / "images/example3.png"

# the weights can be downloaded from: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
checkpoint = package_path.parents[0] / "checkpoint/sam_vit_h_4b8939.pth" 

# Write filtering=True to only identify sections
masks = automatic_identification(image1, checkpoint=checkpoint, compress=True, apply_filtering=True)
print(f"Number of masks identified: {len(masks)}")

Compressing the image...
Image is compressed.
Loaded cached masks.
Filtering masks...
Most common largest cluster size: 94 out of 125 total masks.
Chosen parameters: eps=870.0, min_samples=1
Filtering completed with chosen parameters: (np.float64(870.0), 1)


interactive(children=(IntSlider(value=0, description='index', max=93), Output()), _dom_classes=('widget-intera…

Number of masks identified: 94


In [4]:
from onnx_export import install_and_export_sam_onnx

output_onnx_path = package_path / "onnx_model_.onnx"
quantized_onnx_path = package_path / "onnx_model_quantized.onnx"

final_path = install_and_export_sam_onnx(
    checkpoint=checkpoint,
    output_onnx=output_onnx_path,
    model_type="vit_h",
    return_single_mask=True,
    opset=17,
    quantize_out=quantized_onnx_path,  # here I did op_types_to_quantize=["MatMul", "Gemm"], which reduces quantization highly;
    # to mazimize quantization, do optmize_model = True instead
    gelu_approximate=False,
    use_stability_score=False,
    return_extra_metrics=False,
)
print("ONNX model saved to:", final_path)

[Info] Installing missing package 'onnxruntime-tools'...
[Info] Loading SAM model from checkpoint...


/opt/anaconda3/envs/section_identification/lib/python3.9/site-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.lo

[Info] Exporting ONNX model to '/Users/fredericoaraujo/Documents/section_identification/onnx_model_.onnx'...


[Info] ONNX export completed: /Users/fredericoaraujo/Documents/section_identification/onnx_model_.onnx
[Info] Quantizing model => '/Users/fredericoaraujo/Documents/section_identification/onnx_model_quantized.onnx'...
[Info] Quantization completed.
[Info] Checking exported model with onnxruntime (CPU)...
[Success] Model 'onnx_model_quantized.onnx' runs successfully with ONNXRuntime.
ONNX model saved to: /Users/fredericoaraujo/Documents/section_identification/onnx_model_quantized.onnx


In [40]:
import cv2
import numpy as np
import onnxruntime as ort
import time
from section_identification.create_embedding import create_embedding_if_needed

# ==========================
# 1) Configuration
# ==========================
IMAGE_PATH = image1
EMBEDDING_PATH = package_path / "images/example1_embedding.npy"
MODEL_PATH = package_path / "onnx_model_quantized.onnx"

# ==========================
# 2) Load image & embedding
# ==========================
embedding_path = create_embedding_if_needed(
    IMAGE_PATH,
    checkpoint=checkpoint,
    model_type="vit_h",
    device="cpu"
)

image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise ValueError(f"[Error] Could not read image at {IMAGE_PATH}")

embedding = np.load(str(EMBEDDING_PATH))
if embedding.ndim == 3:
    embedding = np.expand_dims(embedding, axis=0)  # => (1, C, H_e, W_e)

orig_h, orig_w = image.shape[:2]
print(f"[Info] Original image shape: (height={orig_h}, width={orig_w})")

# ==========================
# 3) Create ONNX session
# ==========================
session = ort.InferenceSession(str(MODEL_PATH))
print("[Info] ONNX model loaded.")

# ==========================
# 4) SAM scale logic
# => same as handleImageScale() from the JS snippet
# ==========================
LONG_SIDE = 1024
long_side = max(orig_h, orig_w)
samScale = float(LONG_SIDE) / float(long_side)

sam_new_w = int(round(orig_w * samScale))
sam_new_h = int(round(orig_h * samScale))
print(f"[Info] SAM new size => w={sam_new_w}, h={sam_new_h}, scale={samScale:.4f}")

# ==========================
# 5) Prepare model feeds
# => With an extra “padding point” at (0,0) with label=-1,
#    matching the JS code’s approach.
# ==========================
def prepare_inputs(mouse_x, mouse_y):
    """
    Convert mouse coords from original image space => 1024-based (SAM),
    add an extra dummy point with label=-1, return feed dictionary.
    """
    # Scale the main point
    x_scaled = mouse_x * samScale
    y_scaled = mouse_y * samScale

    # We'll store 2 points total: 1 actual click, 1 dummy
    # shape => (1, 2, 2), labels => (1,2)
    point_coords = np.zeros((1, 2, 2), dtype=np.float32)
    point_labels = np.zeros((1, 2), dtype=np.float32)

    # Fill in the real point
    point_coords[0, 0, 0] = x_scaled
    point_coords[0, 0, 1] = y_scaled
    point_labels[0, 0] = 1.0  # (positive label)

    # Fill in the dummy point => (0,0) with label = -1
    point_coords[0, 1, 0] = 0.0
    point_coords[0, 1, 1] = 0.0
    point_labels[0, 1]    = -1.0

    # No previous mask => mask_input=0’s, has_mask_input=0
    mask_input   = np.zeros((1,1,256,256), dtype=np.float32)
    has_mask_input = np.array([0], dtype=np.float32)

    # original image size => shape (2,) float
    orig_im_size = np.array([orig_h, orig_w], dtype=np.float32)

    feeds = {
        "image_embeddings": embedding,    # (1, C, H_e, W_e)
        "point_coords": point_coords,     # (1,2,2)
        "point_labels": point_labels,     # (1,2)
        "mask_input": mask_input,
        "has_mask_input": has_mask_input,
        "orig_im_size": orig_im_size,
    }
    return feeds

# ==========================
# 6) Process final mask
# => If your model’s final output is (1,1,H,W) at index=0, we threshold & upsample
# => Or if it’s already full-res, skip upsample
# => If it’s 256×256 low-res, do the 2-step upsample.
# ==========================
def process_mask(mask_raw):
    # shape => (1,1,256,256) if "low_res_masks"
    # shape => (1,1, someLargeH, someLargeW) if "masks"
    mask_np = np.squeeze(mask_raw)  # remove (1,1)
    print("[Debug] mask_np shape after squeeze:", mask_np.shape)

    if mask_np.shape == (256,256):
        # => 2-step upsample
        mask_sam = cv2.resize(mask_np, (sam_new_w, sam_new_h), interpolation=cv2.INTER_LINEAR)
        mask_full= cv2.resize(mask_sam, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    else:
        # => Possibly it's already final resolution or partial
        # If shape == (sam_new_h, sam_new_w), then just upsample to (orig_h, orig_w)
        # If shape == (orig_h, orig_w), no resize needed
        # We'll do a small check:
        h_m, w_m = mask_np.shape
        if (h_m, w_m) == (sam_new_h, sam_new_w):
            # upsample to (orig_h, orig_w)
            mask_full = cv2.resize(mask_np, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
        elif (h_m, w_m) == (orig_h, orig_w):
            # already final
            mask_full = mask_np
        else:
            print(f"[Warning] Unexpected mask shape: {mask_np.shape}, using default upsample to original.")
            mask_full = cv2.resize(mask_np, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

    # threshold => binary
    _, mask_bin = cv2.threshold(mask_full, 0.0, 1, cv2.THRESH_BINARY)
    mask_bin = mask_bin.astype(np.uint8)

    overlay = np.zeros((orig_h, orig_w, 3), dtype=np.uint8)
    overlay[..., 2] = mask_bin * 255  # red channel
    return overlay

# ==========================
# 7) Mouse callback
# ==========================
last_time = 0
throttle_secs = 0.05

def mouse_callback(event, x, y, flags, param):
    global last_time

    if event == cv2.EVENT_MOUSEMOVE:
        now = time.time()
        if now - last_time < throttle_secs:
            return
        last_time = now

        feeds = prepare_inputs(x, y)
        try:
            # The official JS code uses the first output => results[model.outputNames[0]]
            # So let's do outputs[0] => shape => (1,1,H,W)
            results = session.run(None, feeds)
            mask_raw = results[0]  # e.g. shape => (1,1,256,256) or (1,1,<finalH>,<finalW>)
        except Exception as e:
            print("[Error] ONNX inference:", e)
            return

        overlay = process_mask(mask_raw)

        # Optionally draw a small circle at the mouse location
        blended = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)
        cv2.circle(blended, (x,y), 3, (0,255,255), -1)
        cv2.imshow("Segment Anything", blended)

# ==========================
# 8) Main Loop
# ==========================
cv2.namedWindow("Segment Anything")
cv2.setMouseCallback("Segment Anything", mouse_callback)
cv2.imshow("Segment Anything", image)

print("Move mouse to see real-time segmentations. ESC to quit.")
while True:
    k = cv2.waitKey(1)
    if k == 27:  # ESC
        break

cv2.destroyAllWindows()

[Info] Creating embedding for /Users/fredericoaraujo/Documents/section_identification/images/example1.png with checkpoint /Users/fredericoaraujo/Documents/checkpoint/sam_vit_h_4b8939.pth...


[ WARN:0@14853.939] global grfmt_png.cpp:695 read_chunk chunk data is too large
/opt/anaconda3/envs/section_identification/lib/python3.9/site-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on Git

[Info] Created embedding file: /Users/fredericoaraujo/Documents/section_identification/images/example1_embedding.npy


[ WARN:0@14864.347] global grfmt_png.cpp:695 read_chunk chunk data is too large


[Info] Original image shape: (height=5000, width=4916)
[Info] ONNX model loaded.
[Info] SAM new size => w=1007, h=1024, scale=0.2048
Move mouse to see real-time segmentations. ESC to quit.
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mask_np shape after squeeze: (5000, 4916)
[Debug] mas

In [41]:
import cv2
import numpy as np
import onnxruntime as ort
import time
from section_identification.manual_detector import create_embedding_if_needed

# ------------------------------
# Configuration - update these paths
# ------------------------------
IMAGE_PATH = image1                   # Path to your image
EMBEDDING_PATH = package_path / "images/example1_embedding.npy"      # Path to your precomputed image embedding
MODEL_PATH = package_path / "onnx_model_quantized.onnx"   # Path to the quantized ONNX model

# ------------------------------
# Embedding
# ------------------------------

embedding_path = create_embedding_if_needed(
            image1,
            checkpoint=checkpoint,
            model_type="vit_h",
            device="cpu"
        )

# ------------------------------
# Load the image and embedding
# ------------------------------
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise ValueError(f"Error loading image at {IMAGE_PATH}")

# The precomputed embedding should be saved with a batch dimension.
# If needed, adjust dimensions so that embedding.shape == (1, C, H, W)
embedding = np.load(EMBEDDING_PATH)
if embedding.ndim == 3:
    embedding = np.expand_dims(embedding, axis=0)

# ------------------------------
# Create ONNX Runtime session
# ------------------------------
session = ort.InferenceSession(MODEL_PATH)

# ------------------------------
# Compute scaling factor
# ------------------------------
# SAM was run on a resized image with its longest side equal to 1024.
# Compute a scaling factor so that mouse (x,y) coordinates (in the original image)
# can be mapped to the resized coordinate space.
orig_h, orig_w = image.shape[:2]
sam_scale = max(orig_w, orig_h) / 1024.0

# ------------------------------
# Helper: Prepare ONNX inputs
# ------------------------------
def prepare_inputs(x, y):
    """
    Given a click coordinate (x,y) in the original image coordinate system,
    convert it to the scaled coordinate system and prepare the input dictionary.
    """
    # Convert the click coordinate to the resized coordinate space
    x_scaled = x / sam_scale
    y_scaled = y / sam_scale

    # Build point input tensors (for one positive click)
    # Updated to have shape [1, 1, 2] as required by the model
    point_coords = np.array([[[x_scaled, y_scaled]]], dtype=np.float32)  # shape (1,1,2)
    # Also update point_labels to have shape [1, 1]
    point_labels = np.array([[1]], dtype=np.float32)

    # SAM ONNX model also expects a mask input (and a flag) even if no prior mask is provided.
    # These are typically zeros.
    # NOTE: The shape (1,1,256,256) is used in the original export code.
    mask_input = np.zeros((1, 1, 256, 256), dtype=np.float32)
    has_mask_input = np.array([0], dtype=np.float32)

    # The model expects the original image size as a float tensor [height, width]
    orig_im_size = np.array([orig_h, orig_w], dtype=np.float32)

    # Return the feed dictionary.
    feeds = {
        "image_embeddings": embedding,
        "point_coords": point_coords,
        "point_labels": point_labels,
        "mask_input": mask_input,
        "has_mask_input": has_mask_input,
        "orig_im_size": orig_im_size,
    }
    return feeds

# ------------------------------
# Helper: Process model output mask
# ------------------------------
def process_mask(mask_raw, target_size):
    """
    Given the raw model output (assumed shape [1, 1, H, W]),
    squeeze it, resize to the target size (width, height),
    threshold to create a binary mask, and return a colored overlay.
    """
    # Remove extra dimensions; now shape is (H, W)
    mask = np.squeeze(mask_raw)
    # Resize mask to match the image size
    mask_resized = cv2.resize(mask, target_size, interpolation=cv2.INTER_LINEAR)
    # Threshold the mask (you can adjust threshold as needed)
    _, mask_binary = cv2.threshold(mask_resized, 0.0, 1, cv2.THRESH_BINARY)
    mask_binary = mask_binary.astype(np.uint8)
    # Create a colored overlay (red color)
    overlay = np.zeros((target_size[1], target_size[0], 3), dtype=np.uint8)
    overlay[:, :, 2] = mask_binary * 255  # red channel
    return overlay

# ------------------------------
# Mouse callback
# ------------------------------
last_inference_time = 0
inference_interval = 0.05  # seconds between inferences to throttle updates

def mouse_callback(event, x, y, flags, param):
    global last_inference_time

    if event == cv2.EVENT_MOUSEMOVE:
        current_time = time.time()
        # Throttle the inference to not run too often
        if current_time - last_inference_time < inference_interval:
            return
        last_inference_time = current_time

        # Prepare model inputs from the current (x,y) mouse coordinate.
        feeds = prepare_inputs(x, y)
        try:
            # Run the ONNX model
            outputs = session.run(None, feeds)
        except Exception as e:
            print("Error during ONNX inference:", e)
            return

        # Assume that the first output is the predicted mask.
        mask_raw = outputs[2]  # e.g., shape (1,1,H_mask,W_mask)
        # Process the mask and create an overlay.
        overlay = process_mask(mask_raw, (orig_w, orig_h))
        # Blend the original image with the overlay.
        blended = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)
        cv2.imshow("Segment Anything", blended)

# ------------------------------
# Set up OpenCV window and mouse callback
# ------------------------------
cv2.namedWindow("Segment Anything")
cv2.setMouseCallback("Segment Anything", mouse_callback)

# Initially display the image
cv2.imshow("Segment Anything", image)
print("Move your mouse over the window to update the segmentation mask. Press ESC to exit.")

# Main loop – wait for the ESC key to exit.
while True:
    key = cv2.waitKey(1)
    if key == 27:  # ESC key
        break

cv2.destroyAllWindows()

[Info] Embedding file already exists for '/Users/fredericoaraujo/Documents/section_identification/images/example1.png': /Users/fredericoaraujo/Documents/section_identification/images/example1_embedding.npy
Move your mouse over the window to update the segmentation mask. Press ESC to exit.


[ WARN:0@14888.781] global grfmt_png.cpp:695 read_chunk chunk data is too large


In [ ]:
# DEBUG VERSION

import cv2
import numpy as np
import onnxruntime as ort
import time
from section_identification.manual_detector import create_embedding_if_needed

# 1) Configuration
IMAGE_PATH = image1
EMBEDDING_PATH = package_path / "images/example2_embedding.npy"
MODEL_PATH = package_path / "onnx_model_quantized.onnx"

# 2) Embedding
embedding_path = create_embedding_if_needed(
    image1,
    checkpoint=checkpoint,
    model_type="vit_h",
    device="cpu"
)

# 3) Load image + embedding
image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise ValueError(f"Error loading image: {IMAGE_PATH}")
orig_h, orig_w = image.shape[:2]

embedding = np.load(EMBEDDING_PATH)
if embedding.ndim == 3:
    embedding = np.expand_dims(embedding, axis=0)

# 4) ONNX session
session = ort.InferenceSession(str(MODEL_PATH))

# 5) aspect-ratio resizing
if orig_w > orig_h:
    sam_new_w = 1024
    scale = 1024.0 / orig_w
    sam_new_h = int(round(orig_h * scale))
else:
    sam_new_h = 1024
    scale = 1024.0 / orig_h
    sam_new_w = int(round(orig_w * scale))

print("[Debug] Original (W,H):", (orig_w, orig_h))
print("[Debug] Resized (W,H):", (sam_new_w, sam_new_h))
print("[Debug] Scale factor:", scale)

# 6) prepare_inputs
def prepare_inputs(x, y):
    x_scaled = x * (sam_new_w / orig_w)
    y_scaled = y * (sam_new_h / orig_h)
    print(f"[Debug] Mouse=({x},{y}), scaled=({x_scaled:.2f},{y_scaled:.2f})")

    point_coords = np.array([[[x_scaled, y_scaled]]], dtype=np.float32)
    point_labels = np.array([[1]], dtype=np.float32)
    mask_input   = np.zeros((1, 1, 256, 256), dtype=np.float32)
    has_mask_input = np.array([0], dtype=np.float32)
    orig_im_size = np.array([orig_h, orig_w], dtype=np.float32)

    feeds = {
        "image_embeddings": embedding,
        "point_coords": point_coords,
        "point_labels": point_labels,
        "mask_input": mask_input,
        "has_mask_input": has_mask_input,
        "orig_im_size": orig_im_size,
    }
    return feeds

# 7) process_mask with bounding box debug
def process_mask(mask_raw):
    # shape => (1,1,256,256)
    print("[Debug] mask_raw shape:", mask_raw.shape)
    mask_256 = np.squeeze(mask_raw)  # (256,256)
    print("[Debug] after squeeze =>", mask_256.shape)

    # Upsample to (sam_new_w, sam_new_h)
    mask_sam = cv2.resize(
        mask_256,
        (sam_new_w, sam_new_h),
        interpolation=cv2.INTER_LINEAR
    )
    print("[Debug] mask_sam shape:", mask_sam.shape)

    # Then upsample to (orig_w, orig_h)
    mask_full = cv2.resize(
        mask_sam,
        (orig_w, orig_h),
        interpolation=cv2.INTER_LINEAR
    )
    print("[Debug] mask_full shape:", mask_full.shape)

    # threshold
    _, mask_bin = cv2.threshold(mask_full, 0.0, 1, cv2.THRESH_BINARY)
    mask_bin = mask_bin.astype(np.uint8)

    overlay = np.zeros((orig_h, orig_w, 3), dtype=np.uint8)
    overlay[..., 2] = mask_bin * 255  # red

    # bounding box
    coords = cv2.findNonZero(mask_bin)
    if coords is not None:
        min_xy = coords.min(axis=0)[0]
        max_xy = coords.max(axis=0)[0]
        x_min, y_min = min_xy
        x_max, y_max = max_xy
        print(f"[Debug] BBox => ({x_min},{y_min}) -> ({x_max},{y_max})")
        # draw it in cyan
        cv2.rectangle(overlay, (x_min,y_min), (x_max,y_max), (255,255,0), 2)

    return overlay, coords

# 8) Mouse callback
last_inference_time = 0
inference_interval = 0.1

def mouse_callback(event, x, y, flags, param):
    global last_inference_time

    if event == cv2.EVENT_MOUSEMOVE:
        now = time.time()
        if now - last_inference_time < inference_interval:
            return
        last_inference_time = now

        feeds = prepare_inputs(x, y)
        try:
            outputs = session.run(None, feeds)
        except Exception as e:
            print("[Error] ONNX inference:", e)
            return

        # we assume the low_res_masks output is outputs[2], or whichever index
        mask_raw = outputs[1]  # confirm you are using the correct index if you changed the script
        overlay, coords = process_mask(mask_raw)

        # Draw a small circle at the mouse location
        blend = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)
        cv2.circle(blend, (x,y), 4, (0,255,255), -1)

        # If we have coords, find bounding box center, print offset
        if coords is not None:
            min_xy = coords.min(axis=0)[0]
            max_xy = coords.max(axis=0)[0]
            x_c = int((min_xy[0] + max_xy[0])/2)
            y_c = int((min_xy[1] + max_xy[1])/2)
            # Mark the center in green
            cv2.circle(blend, (x_c,y_c), 4, (0,255,0), -1)

            dx = x - x_c
            dy = y - y_c
            print(f"[Debug] Offset => mouse=({x},{y}) - mask_center=({x_c},{y_c}) => ({dx},{dy})")

        cv2.imshow("Segment Anything", blend)

# 9) Main
cv2.namedWindow("Segment Anything")
cv2.setMouseCallback("Segment Anything", mouse_callback)
cv2.imshow("Segment Anything", image)
print("Move your mouse; watch debug logs. Press ESC to quit.")

while True:
    k = cv2.waitKey(1)
    if k == 27:  # ESC
        break

cv2.destroyAllWindows()

[Info] Embedding file already exists for '/Users/fredericoaraujo/Documents/section_identification/images/example1.png': /Users/fredericoaraujo/Documents/section_identification/images/example1_embedding.npy
[Debug] Original (W,H): (4916, 5000)
[Debug] Resized (W,H): (1007, 1024)
[Debug] Scale factor: 0.2048


[ WARN:0@12851.677] global grfmt_png.cpp:695 read_chunk chunk data is too large


Move your mouse; watch debug logs. Press ESC to quit.
[Debug] Mouse=(3431,3270), scaled=(702.81,669.70)
[Debug] mask_raw shape: (1, 1)
[Debug] after squeeze => ()
[Debug] mask_sam shape: (1024, 1007)
[Debug] mask_full shape: (5000, 4916)
[Debug] BBox => (0,0) -> (4915,4999)
[Debug] Offset => mouse=(3431,3270) - mask_center=(2457,2499) => (974,771)
[Debug] Mouse=(3490,2770), scaled=(714.90,567.30)
[Debug] mask_raw shape: (1, 1)
[Debug] after squeeze => ()
[Debug] mask_sam shape: (1024, 1007)
[Debug] mask_full shape: (5000, 4916)
[Debug] BBox => (0,0) -> (4915,4999)
[Debug] Offset => mouse=(3490,2770) - mask_center=(2457,2499) => (1033,271)
[Debug] Mouse=(3483,2766), scaled=(713.46,566.48)
[Debug] mask_raw shape: (1, 1)
[Debug] after squeeze => ()
[Debug] mask_sam shape: (1024, 1007)
[Debug] mask_full shape: (5000, 4916)
[Debug] BBox => (0,0) -> (4915,4999)
[Debug] Offset => mouse=(3483,2766) - mask_center=(2457,2499) => (1026,267)
[Debug] Mouse=(3040,2078), scaled=(622.72,425.57)
[Debug